# 03b — Подробный анализ лучших параметров

> **Статус сохранённого анализа:** `outputs/besy/run_01/best_result_analysis/` устарел. Он рассчитан на прежний неполный корпус текста (1 086 475 символов); его позиции и выводы нельзя переносить на текущий полный корпус.

Запускает matching с лучшими параметрами из экспериментов и сохраняет детальные артефакты для анализа ошибок.

**Параметры:** α=0.0, β=1.0, γ=0.0 (word TF-IDF only), group=60s, window=400, threshold=0.20

In [1]:
import pickle, json, os
from pathlib import Path
from datetime import datetime
import numpy as np

PROJECT_ROOT = Path(os.environ.get(
    "SPARK_ROOT",
    "/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit"
))
RUN_DIR = PROJECT_ROOT / "outputs/besy/run_01"
ANALYSIS_DIR = RUN_DIR / "best_result_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

def to_json_safe(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_json_safe(x) for x in obj]
    return obj

print("Базовые импорты готовы")

Базовые импорты готовы


In [2]:
# Загружаем данные
with open(RUN_DIR / "normalized_text.pkl", "rb") as f:
    text_data = pickle.load(f)
normalized_text = text_data["normalized_text"]
section_boundaries = text_data["section_boundaries"]

with open(RUN_DIR / "audio_segments.pkl", "rb") as f:
    audio_data = pickle.load(f)
audio_segments = audio_data["audio_segments"]
timeline = audio_data["timeline"]
total_duration = audio_data["total_duration"]

print(f"Текст: {len(normalized_text):,} симв, Аудио: {len(audio_segments):,} сегментов")

Текст: 1,086,475 симв, Аудио: 35,260 сегментов


In [3]:
# Копируем проверенные функции из ноутбука 03

def build_audio_groups(segments, group_size_sec=60, min_group_sec=10):
    groups = []
    buf_text, buf_start, buf_end, buf_file, buf_idx = [], None, None, None, None
    for i, s in enumerate(segments):
        if buf_start is None:
            buf_start = s["global_start"]; buf_file = s["file"]; buf_idx = s["file_index"]
        buf_text.append(s["text"]); buf_end = s["global_end"]
        dur = buf_end - buf_start
        next_idx = i + 1 if i + 1 < len(segments) else None
        file_boundary = (next_idx is None) or (segments[next_idx]["file_index"] != buf_idx)
        if file_boundary and dur < min_group_sec: continue
        if dur >= group_size_sec or file_boundary:
            groups.append({"audio_start": buf_start, "audio_end": buf_end,
                "duration": dur, "text": " ".join(buf_text),
                "file": buf_file, "file_index": buf_idx, "segment_count": len(buf_text)})
            buf_text, buf_start = [], None
    if buf_text:
        groups.append({"audio_start": buf_start, "audio_end": buf_end,
            "duration": buf_end - buf_start, "text": " ".join(buf_text),
            "file": buf_file, "file_index": buf_idx, "segment_count": len(buf_text)})
    return groups

def build_text_windows(text, window_size=400, step=100):
    windows = []
    for start in range(0, len(text) - window_size, step):
        windows.append({"char_start": start, "char_end": start + window_size,
                        "text": text[start:start + window_size]})
    if windows and windows[-1]["char_end"] < len(text):
        windows.append({"char_start": len(text) - window_size, "char_end": len(text),
                        "text": text[-window_size:]})
    return windows

def resolve_section(char_pos, boundaries):
    for start, end, title in boundaries:
        if start <= char_pos <= end: return title
    return "?"

print("Функции подготовки готовы")

Функции подготовки готовы


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Строим аудиогруппы и окна
GROUP_SIZE = 60
WINDOW_SIZE = 400
WINDOW_STEP = 100

audio_groups = build_audio_groups(audio_segments, group_size_sec=GROUP_SIZE)
text_windows = build_text_windows(normalized_text, window_size=WINDOW_SIZE, step=WINDOW_STEP)

group_texts = [g["text"] for g in audio_groups]
window_texts = [w["text"] for w in text_windows]

print(f"Групп: {len(audio_groups)}, Окон: {len(text_windows)}")

Групп: 2088, Окон: 10862


In [5]:
# Word TF-IDF matching
print("TF-IDF…", end=" ", flush=True)
vec = TfidfVectorizer(analyzer='word', max_features=50000)
all_texts = window_texts + group_texts
tfidf = vec.fit_transform(all_texts)
W = tfidf[:len(window_texts)]
G = tfidf[len(window_texts):]

# Для каждой группы: топ-3 кандидата + их similarity
TOP_K = 3
all_candidates = []
for i in range(G.shape[0]):
    sims = cosine_similarity(G[i], W)[0]
    top = np.argpartition(sims, -TOP_K)[-TOP_K:]
    top = top[np.argsort(sims[top])[::-1]]
    all_candidates.append([(int(t), float(sims[t])) for t in top])
print("OK")

TF-IDF… OK


In [6]:
# Принимаем решение: лучший кандидат должен пройти порог
THRESHOLD = 0.20

audio_map = []
unmatched = []

for i, g in enumerate(audio_groups):
    best_idx, best_score = all_candidates[i][0]
    
    if best_score < THRESHOLD:
        unmatched.append({
            "group_index": i,
            "audio_start": g["audio_start"],
            "audio_end": g["audio_end"],
            "duration": g["duration"],
            "file": g["file"],
            "file_index": g["file_index"],
            "text": g["text"][:300],
            "best_score": round(best_score, 4),
            "candidates": all_candidates[i],
        })
    else:
        audio_map.append({
            "group_index": i,
            "audio_start": g["audio_start"],
            "audio_end": g["audio_end"],
            "char_start": text_windows[best_idx]["char_start"],
            "char_end": text_windows[best_idx]["char_end"],
            "file": g["file"],
            "file_index": g["file_index"],
            "score": round(best_score, 4),
            "section": resolve_section(text_windows[best_idx]["char_start"], section_boundaries),
            "audio_text": g["text"][:200],
            "matched_text": normalized_text[text_windows[best_idx]["char_start"]:text_windows[best_idx]["char_start"]+200],
        })

print(f"Matched: {len(audio_map)}, Unmatched: {len(unmatched)}")
print(f"Unmatched доля: {100*len(unmatched)/len(audio_groups):.1f}%")

Matched: 1883, Unmatched: 205
Unmatched доля: 9.8%


In [7]:
# ─── АНАЛИЗ ОШИБОК ───

# 1. Нарушения монотонности
print("=" * 70)
print("1. НАРУШЕНИЯ МОНОТОННОСТИ")
print("=" * 70)

violations = []
for i in range(1, len(audio_map)):
    if audio_map[i]["char_start"] < audio_map[i-1]["char_start"]:
        jump_back = audio_map[i-1]["char_start"] - audio_map[i]["char_start"]
        violations.append({
            "at_group": i,
            "prev_group": audio_map[i-1]["group_index"],
            "curr_group": audio_map[i]["group_index"],
            "prev_section": audio_map[i-1]["section"],
            "curr_section": audio_map[i]["section"],
            "prev_pos": audio_map[i-1]["char_start"],
            "curr_pos": audio_map[i]["char_start"],
            "jump_back_chars": int(jump_back),
            "prev_time": audio_map[i-1]["audio_start"],
            "curr_time": audio_map[i]["audio_start"],
            "prev_score": audio_map[i-1]["score"],
            "curr_score": audio_map[i]["score"],
        })

print(f"Всего нарушений: {len(violations)} из {len(audio_map)-1} переходов ({100*len(violations)/(len(audio_map)-1):.1f}%)")

# Группируем по величине скачка
print(f"\nРаспределение величины скачка назад (символов):")
jumps = [v["jump_back_chars"] for v in violations]
bins = [0, 1000, 10000, 100000, 500000, 1000000]
for lo, hi in zip(bins[:-1], bins[1:]):
    count = sum(1 for j in jumps if lo <= j < hi)
    if count:
        print(f"  {lo:>7,} – {hi:>7,}: {count}")

# Показываем крупнейшие скачки
violations_by_size = sorted(violations, key=lambda x: x["jump_back_chars"], reverse=True)
print(f"\nКрупнейшие скачки назад:")
for v in violations_by_size[:10]:
    prev_mm = int(v["prev_time"] // 60); prev_ss = int(v["prev_time"] % 60)
    curr_mm = int(v["curr_time"] // 60); curr_ss = int(v["curr_time"] % 60)
    print(f"  Скачок {v['jump_back_chars']:>9,} симв назад")
    print(f"    [{prev_mm:3d}:{prev_ss:02d}] → [{curr_mm:3d}:{curr_ss:02d}]")
    print(f"    Было: {v['prev_section']} (поз.{v['prev_pos']:,}), score={v['prev_score']:.3f}")
    print(f"    Стало: {v['curr_section']} (поз.{v['curr_pos']:,}), score={v['curr_score']:.3f}")

with open(ANALYSIS_DIR / "violations.json", "w") as f:
    json.dump(to_json_safe(violations), f, indent=2, ensure_ascii=False)

1. НАРУШЕНИЯ МОНОТОННОСТИ
Всего нарушений: 5 из 1882 переходов (0.3%)

Распределение величины скачка назад (символов):
        0 –   1,000: 1
    1,000 –  10,000: 3
  500,000 – 1,000,000: 1

Крупнейшие скачки назад:
  Скачок   985,700 симв назад
    [  1:01] → [  2:03]
    Было: Часть третья / Глава седьмая (поз.985,700), score=0.613
    Стало: Часть первая / Глава первая (поз.0), score=0.799
  Скачок     2,700 симв назад
    [1074:26] → [1075:30]
    Было: Часть вторая / Глава шестая (поз.547,900), score=0.232
    Стало: Часть вторая / Глава шестая (поз.545,200), score=0.790
  Скачок     2,000 симв назад
    [1861:44] → [1862:47]
    Было: Часть третья / Глава шестая (поз.933,900), score=0.817
    Стало: Часть третья / Глава шестая (поз.931,900), score=0.371
  Скачок     1,200 симв назад
    [1702:11] → [1706:16]
    Было: Часть третья / Глава четвертая (поз.845,000), score=0.769
    Стало: Часть третья / Глава четвертая (поз.843,800), score=0.203
  Скачок       300 симв назад
    [19

In [8]:
# 2. Анализ unmatched групп
print("\n" + "=" * 70)
print("2. НЕСОПОСТАВЛЕННЫЕ ГРУППЫ")
print("=" * 70)

# По файлам
from collections import Counter
by_file = Counter(u["file"] for u in unmatched)
print(f"\nРаспределение по файлам:")
for fname, count in by_file.most_common():
    print(f"  {count:>3} unmatched — {fname[:60]}")

# По scores
scores = [u["best_score"] for u in unmatched]
print(f"\nРаспределение best_score:")
print(f"  min={min(scores):.4f}  mean={np.mean(scores):.4f}  median={np.median(scores):.4f}  max={max(scores):.4f}")

# Показываем текст unmatched — первые 30
print(f"\nПримеры unmatched (первые 30):")
for u in unmatched[:30]:
    mm = int(u["audio_start"] // 60); ss = int(u["audio_start"] % 60)
    cand_scores = [f"{s:.3f}" for _, s in u["candidates"]]
    print(f"  [{mm:3d}:{ss:02d}] score={u['best_score']:.3f} | {u['file'][:35]} | \"{u['text'][:120]}\"")

with open(ANALYSIS_DIR / "unmatched.json", "w") as f:
    json.dump(to_json_safe(unmatched), f, indent=2, ensure_ascii=False)


2. НЕСОПОСТАВЛЕННЫЕ ГРУППЫ

Распределение по файлам:
   42 unmatched — 101 Вместо введения - несколько подробностей из биографии мн
   23 unmatched — 301 Праздник. Отдел первый.mp3
   19 unmatched — 210 Флибустьеры. Роковое утро.mp3
   16 unmatched — 302 Окончание праздника.mp3
   15 unmatched — 205 Пред праздником.mp3
   11 unmatched — 103 Чужие грехи.mp3
    9 unmatched — 201 Ночь.mp3
    9 unmatched — 304 Последнее решение.mp3
    8 unmatched — 209 Степана Трофимовича описали.mp3
    7 unmatched — 102 Принц Гарри. Сватовство.mp3
    7 unmatched — 204 Все в ожидании.mp3
    7 unmatched — 307 Последнее странствование Степана Трофимовича.mp3
    6 unmatched — 105 Премудрый змий.mp3
    6 unmatched — 207 У наших.mp3
    6 unmatched — 305 Путешественница.mp3
    4 unmatched — 206 Петр Степанович в хлопотах.mp3
    3 unmatched — 203 Поединок.mp3
    2 unmatched — 309 У Тихона.mp3
    1 unmatched — 104 Хромоножка.mp3
    1 unmatched — 202 Ночь (продолжение).mp3
    1 unmatched — 208 Иван-

In [9]:
# 3. Стыки файлов
print("\n" + "=" * 70)
print("3. СТЫКИ ФАЙЛОВ")
print("=" * 70)

boundary_issues = []
for i in range(1, len(audio_map)):
    if audio_map[i]["file_index"] != audio_map[i-1]["file_index"]:
        prev_end = audio_map[i-1]["char_end"]
        curr_start = audio_map[i]["char_start"]
        gap = curr_start - prev_end
        boundary_issues.append({
            "transition": f"{audio_map[i-1]['file'][:40]} → {audio_map[i]['file'][:40]}",
            "prev_char_end": prev_end,
            "curr_char_start": curr_start,
            "gap_chars": int(gap),
            "prev_section": audio_map[i-1]["section"],
            "curr_section": audio_map[i]["section"],
        })

gaps = [b["gap_chars"] for b in boundary_issues]
print(f"Границ файлов: {len(boundary_issues)}")
print(f"Разрывы: min={min(gaps):,}  mean={np.mean(gaps):.0f}  median={np.median(gaps):.0f}  max={max(gaps):,}")
print(f"Положительные (вперёд): {sum(1 for g in gaps if g > 0)}")
print(f"Отрицательные (назад): {sum(1 for g in gaps if g < 0)}")

print(f"\nКрупнейшие разрывы на стыках:")
boundary_by_size = sorted(boundary_issues, key=lambda x: abs(x["gap_chars"]), reverse=True)
for b in boundary_by_size[:10]:
    direction = "→" if b["gap_chars"] >= 0 else "←"
    print(f"  {b['gap_chars']:>9,} симв {direction}  {b['transition']}")
    print(f"    {b['prev_section']} → {b['curr_section']}")

with open(ANALYSIS_DIR / "boundary_issues.json", "w") as f:
    json.dump(to_json_safe(boundary_issues), f, indent=2, ensure_ascii=False)


3. СТЫКИ ФАЙЛОВ
Границ файлов: 23
Разрывы: min=-200  mean=130  median=100  max=1,400
Положительные (вперёд): 14
Отрицательные (назад): 3

Крупнейшие разрывы на стыках:
      1,400 симв →  304 Последнее решение.mp3 → 305 Путешественница.mp3
    Часть третья / Глава четвертая → Часть третья / Глава пятая
        300 симв →  210 Флибустьеры. Роковое утро.mp3 → 301 Праздник. Отдел первый.mp3
    Часть вторая / Глава десятая → Часть третья / Глава первая
       -200 симв ←  102 Принц Гарри. Сватовство.mp3 → 103 Чужие грехи.mp3
    Часть первая / Глава вторая → Часть первая / Глава вторая
        200 симв →  105 Премудрый змий.mp3 → 201 Ночь.mp3
    Часть первая / Глава пятая → Часть вторая / Глава первая
        200 симв →  203 Поединок.mp3 → 204 Все в ожидании.mp3
    Часть вторая / Глава третья → Часть вторая / Глава четвертая
        200 симв →  205 Пред праздником.mp3 → 206 Петр Степанович в хлопотах.mp3
    Часть вторая / Глава пятая → Часть вторая / Глава шестая
        200 симв →  2

In [10]:
# 4. Карта matching: время аудио → позиция в тексте
print("\n" + "=" * 70)
print("4. ОБЩАЯ КАРТА")
print("=" * 70)

times = [m["audio_start"] / 3600 for m in audio_map]
positions = [m["char_start"] for m in audio_map]

# Плотность: chars per second
total_chars = positions[-1] - positions[0] if positions else 0
total_time = times[-1] - times[0] if times else 0
chars_per_sec = total_chars / total_time if total_time > 0 else 0
print(f"Покрытие текста: {positions[0]:,} → {positions[-1]:,} ({total_chars:,} симв)")
print(f"Покрытие аудио: {times[0]:.1f}ч → {times[-1]:.1f}ч ({total_time:.1f}ч)")
print(f"Средняя скорость: {chars_per_sec:.0f} симв/с")

# По частям
from collections import Counter
section_counts = Counter(m["section"] for m in audio_map)
print(f"\nРаспределение по главам:")
for section, count in section_counts.most_common():
    print(f"  {count:>4} групп — {section}")

# Score distribution
scores = [m["score"] for m in audio_map]
print(f"\nРаспределение confidence (TF-IDF similarity):")
for lo, hi in [(0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 0.9), (0.9, 1.0)]:
    count = sum(1 for s in scores if lo <= s < hi)
    print(f"  [{lo:.1f}–{hi:.1f}): {count:>5} групп ({100*count/len(scores):.1f}%)")

# Score by section — где matching хуже всего
section_scores = {}
for m in audio_map:
    sec = m["section"]
    section_scores.setdefault(sec, []).append(m["score"])
print(f"\nГлавы с наихудшим средним score:")
avg_by_section = [(sec, np.mean(ss), len(ss)) for sec, ss in section_scores.items()]
for sec, avg, cnt in sorted(avg_by_section, key=lambda x: x[1])[:10]:
    print(f"  avg={avg:.3f}  n={cnt:>4} — {sec}")


4. ОБЩАЯ КАРТА
Покрытие текста: 985,400 → 1,086,075 (100,675 симв)
Покрытие аудио: 0.0ч → 36.2ч (36.2ч)
Средняя скорость: 2781 симв/с

Распределение по главам:
   132 групп — Часть первая / Глава пятая
   131 групп — Часть вторая / Глава первая
   122 групп — Часть вторая / Глава шестая
   119 групп — Часть первая / Глава третья
   114 групп — Часть первая / Глава вторая
   111 групп — Приложение / Глава девятая. У Тихона
   108 групп — Часть третья / Глава седьмая
    98 групп — Часть первая / Глава четвертая
    91 групп — Часть третья / Глава шестая
    80 групп — Часть третья / Глава пятая
    75 групп — Часть третья / Глава вторая
    73 групп — Часть вторая / Глава вторая
    71 групп — Часть третья / Глава первая
    70 групп — Часть первая / Глава первая
    63 групп — Часть вторая / Глава пятая
    62 групп — Часть вторая / Глава седьмая
    61 групп — Часть третья / Глава третья
    61 групп — Часть третья / Глава четвертая
    60 групп — Часть вторая / Глава четвертая
    5

In [11]:
# 5. Сохраняем всё для дальнейшего анализа

# Полная карта
with open(ANALYSIS_DIR / "audio_map.pkl", "wb") as f:
    pickle.dump(audio_map, f)

# Конфиг
with open(ANALYSIS_DIR / "config.json", "w") as f:
    json.dump({
        "group_size": GROUP_SIZE, "window_size": WINDOW_SIZE,
        "window_step": WINDOW_STEP, "alpha": 0.0, "beta": 1.0, "gamma": 0.0,
        "threshold": THRESHOLD, "tfidf_method": "word",
        "n_groups": len(audio_groups), "n_windows": len(text_windows),
        "n_matched": len(audio_map), "n_unmatched": len(unmatched),
        "n_violations": len(violations),
        "timestamp": datetime.now().isoformat(),
    }, f, indent=2)

# Текстовые примеры для ручной проверки (20 точек)
with open(ANALYSIS_DIR / "manual_check.txt", "w", encoding="utf-8") as f:
    f.write("=== 20 ТОЧЕК ДЛЯ РУЧНОЙ ПРОВЕРКИ ===\n\n")
    rng = np.random.RandomState(123)
    indices = sorted(rng.choice(len(audio_map), min(20, len(audio_map)), replace=False))
    for idx in indices:
        m = audio_map[idx]
        mm, ss = int(m["audio_start"] // 60), int(m["audio_start"] % 60)
        f.write(f"[{mm:3d}:{ss:02d}] {m['file']}\n")
        f.write(f"  Глава: {m['section']}\n")
        f.write(f"  Score: {m['score']:.3f}\n")
        f.write(f"  Аудио: \"{m['audio_text'][:200]}\"\n")
        f.write(f"  Текст: \"{m['matched_text'][:200]}\"\n\n")
    
    # Примеры нарушений
    f.write("\n=== ПРИМЕРЫ НАРУШЕНИЙ МОНОТОННОСТИ ===\n\n")
    for v in violations[:15]:
        f.write(f"Скачок назад на {v['jump_back_chars']:,} симв\n")
        f.write(f"  [{int(v['prev_time']//60):3d}:{int(v['prev_time']%60):02d}] {v['prev_section']} (score={v['prev_score']:.3f})\n")
        f.write(f"  [{int(v['curr_time']//60):3d}:{int(v['curr_time']%60):02d}] {v['curr_section']} (score={v['curr_score']:.3f})\n\n")

print(f"\nАртефакты сохранены в {ANALYSIS_DIR}/")
print(f"  config.json, audio_map.pkl, violations.json, unmatched.json, boundary_issues.json, manual_check.txt")
print(f"\nГотово к анализу.")


Артефакты сохранены в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01/best_result_analysis/
  config.json, audio_map.pkl, violations.json, unmatched.json, boundary_issues.json, manual_check.txt

Готово к анализу.
